# 49 — P2Rank (PDB) / P2Rank (AFDB) / Random patches: pooled, deduplicated corpora

**Third comparison family**, alongside structure-source (37b AFDB / 45 apo / 46 holo) and
pocket-*prediction*: instead of asking "what if we replace the experimental structure", this asks
"what if we replace the experimental (ligand-defined) pocket with an automatically *detected* one,
or with nothing at all (a random patch)". Same population-handling approach throughout: pooled across
the four datasets, cross-dataset deduplicated on `(pdb_id, lig_resname)` (same key/priority as
37b/45/46 — `combined_deleakage_analysis.dedup_cross_dataset`), self-scores (bound ligand vs pocket).

Three conditions:

| | Pocket source | Detection-accuracy metric | Correlation analysis |
|---|---|---|---|
| **PDB P2Rank** | P2Rank pocket prediction run on the experimental structure, best-DCA match kept | DCA, DCC, IoU, recall (all from nb35's sensitivity tables) | yes |
| **AFDB P2Rank** | P2Rank pocket prediction run on the AlphaFold model, best-DCA match kept | DCA, DCC (from nb38) | yes |
| **Patches** | Random surface patch(es) on the experimental structure — no detection involved | none — nothing to measure accuracy against | **no** (§9 in nb50 is score-distribution only) |

"Best-DCA match" (not top-ranked / `rank1`) is used for **both** P2Rank conditions, for
apples-to-apples consistency: nb35's PDB sensitivity tables already pick the single best-matching
predicted pocket per structure (by DCA, over however many pockets P2Rank returned), and nb38's AFDB
`best_dca` strategy does the same thing. Neither is "what a real user would pick without ground
truth" (that would be `rank1`) — both are "how good is the best pocket P2Rank found here", an
upper-bound / oracle-style read, chosen to match the accuracy metric that's actually already
computed on both sides rather than re-deriving PDB `rank1` DCA/DCC from scratch.

Patches are **mean-pooled** across all patches on a pocket's source structure — same "mean-pooled
score = score of the mean-pooled embedding" convention nb42/43/45/46 use for multi-counterpart
conditions (linear in the dot product, so it doesn't matter which order you do it in).

**Inputs:**
- `RESULTS/<ds>/p2rank_pocket_encodings/pocket_reps.pkl` + `RESULTS/benchmark/<ds>_pdb_p2rank_sensitivity.tsv` — PDB P2Rank
- `RESULTS/afdb_p2rank_pockets/nb38_all_scores.tsv` + `pocket_reps.pkl` — AFDB P2Rank (scores/DCA/DCC already computed by nb38)
- `RESULTS/afdb_p2rank/AF-<uniprot>-F1-model_v6.cif_predictions.csv` — raw P2Rank output per UniProt (residue_ids per predicted pocket, in AFDB numbering)
- `RESULTS/afdb_p2rank_pockets/aln_map_cache.pkl` — {afdb_label_seq_id: pdb_resseq} per (entry, uniprot_acc, lig_chain), cached by nb38's own alignment step
- `NATURAL_LIGANDS/RESULTS/pockets/<ds>/pocket_manifest.tsv` — `pocket_residues` (true/observed 6 Å contact residues, JSON [chain,resseq,icode,resname])
- `RESULTS/<ds>/pocket_reps.pkl` — random surface patches (nb30/31)
- `NATURAL_LIGANDS/RESULTS/pockets/<ds>_pocket_reps.pkl` + `mols/bound_<ds>.h5` — experimental pocket / bound ligand embeddings (self-score baseline)
- `RESULTS/afdb_pockets_6A/afdb_pockets_manifest_qc.tsv` — UniProt lookup (same SIFTS mapping as 37b/45/46)

In [2]:
from __future__ import annotations
import json
import pickle
import re
from pathlib import Path
from collections import defaultdict

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NL   = Path.cwd().parent / 'DATA' / 'natural_ligands' / 'RESULTS'
PROJ = Path.cwd().parent / 'DATA' / 'drugclip'
OUT_DIR = PROJ / 'RESULTS/p2rank_patches_pooled'
OUT_DIR.mkdir(exist_ok=True)

DATASETS = ['coach420', 'holo4k', 'pdbbind2020', 'scpdb']
DS_COLORS = {
    'coach420':    '#4C72B0',
    'holo4k':      '#DD8452',
    'pdbbind2020': '#55A868',
    'scpdb':       '#C44E52',
}

# Condition colours from 42_drugclip_benchmark.ipynb's COLOURS dict — same notebook that already
# names these three exact conditions ('pdb_p2rank', 'afdb_p2rank', 'patches').
SOURCE_C      = '#2CA02C'   # 'bound'
PDB_P2RANK_C  = '#E377C2'   # 'pdb_p2rank'  (pink)
AFDB_P2RANK_C = '#D62728'   # 'afdb_p2rank' (red)
PATCH_C       = '#000000'   # 'patches'     (black)

DEDUP_PRIORITY = ['coach420', 'scpdb', 'pdbbind2020', 'holo4k']

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})


def square(fig):
    for ax in fig.axes:
        ax.set_box_aspect(1)
    return fig


def pct_hist(ax, vals, bins, **kwargs):
    vals = np.asarray(vals)
    weights = np.full(len(vals), 100.0 / len(vals)) if len(vals) else None
    return ax.hist(vals, bins=bins, weights=weights, **kwargs)


def dedup_cross_dataset(pp: pd.DataFrame,
                         priority: list[str] = DEDUP_PRIORITY,
                         key_cols: tuple[str, str] = ('pdb_id', 'lig_resname')) -> pd.DataFrame:
    """Same dedup as 37b / 45 / 46 — keep the highest-priority dataset's row for each
    (pdb_id, lig_resname) pair spanning more than one dataset."""
    pdb_col, lig_col = key_cols
    key = pd.Series(
        list(zip(pp[pdb_col].str.lower().str.strip(), pp[lig_col].str.upper())),
        index=pp.index,
    )
    winner = pp.groupby(key)['dataset'].transform(
        lambda s: next(d for d in priority if d in set(s))
    )
    return pp[pp['dataset'] == winner].copy()


def cosine_scores(lig_rep: np.ndarray, lig_mask: np.ndarray, tgt_reps: np.ndarray) -> np.ndarray:
    """lig_rep (6,128), lig_mask (6,) bool, tgt_reps (6,T,128) -> (T,)."""
    per_fold = np.einsum('kd,ktd->kt', lig_rep, tgt_reps)
    n_valid = int(lig_mask.sum())
    if n_valid == 0:
        return np.full(tgt_reps.shape[1], np.nan, dtype=np.float32)
    return (per_fold * lig_mask[:, None]).sum(axis=0) / n_valid


def pocket_cosine(rep_a: np.ndarray, rep_b: np.ndarray) -> float:
    """Mean per-fold cosine similarity between two (6, 128) pocket reps, L2-normalised per fold."""
    norm_a = np.linalg.norm(rep_a, axis=1, keepdims=True)
    norm_b = np.linalg.norm(rep_b, axis=1, keepdims=True)
    a_n = rep_a / (norm_a + 1e-8)
    b_n = rep_b / (norm_b + 1e-8)
    return float(np.mean(np.sum(a_n * b_n, axis=1)))


def load_bound_pocket_reps(ds: str):
    return pickle.load(open(NL / 'pockets' / f'{ds}_pocket_reps.pkl', 'rb'))


def load_mol_reps(ds: str):
    """(manifest_df, mol_reps (N,6,128), fold_masks (N,6) bool) — h5 in lex LMDB-key order."""
    h5_path = NL / 'mols' / f'bound_{ds}.h5'
    mf_path = NL / 'mols' / f'bound_{ds}_manifest.tsv'
    mf = pd.read_csv(mf_path, sep='\t')
    with h5py.File(h5_path, 'r') as f:
        mol_reps = f['mol_reps'][:]
        fold_masks = np.stack([f[f'fold{k}'][:] for k in range(6)], axis=1)
    lex_order = mf['lmdb_key'].astype(str).argsort()
    mf = mf.iloc[lex_order].reset_index(drop=True)
    mol_reps = mol_reps.reshape(len(mol_reps), 6, 128)
    fold_masks = fold_masks.astype(bool)
    return mf, mol_reps, fold_masks


# ── Shared lookups ──────────────────────────────────────────────────────────
mol_lookup = pd.concat(
    [pd.read_csv(NL / f'mols/bound_{ds}_manifest.tsv', sep='\t',
                 usecols=['pocket', 'pdb_id', 'lig_resname']) for ds in DATASETS],
    ignore_index=True,
).drop_duplicates('pocket').set_index('pocket')

afdb_manifest = pd.read_csv(PROJ / 'RESULTS/afdb_pockets_6A/afdb_pockets_manifest_qc.tsv', sep='\t',
                             usecols=['pocket_id', 'uniprot_acc'])
uniprot_lookup = dict(zip(afdb_manifest['pocket_id'], afdb_manifest['uniprot_acc']))

print(f'mol_lookup: {len(mol_lookup):,} pockets  |  uniprot_lookup: {len(uniprot_lookup):,} pockets')

mol_lookup: 17,305 pockets  |  uniprot_lookup: 14,366 pockets


## 1. PDB P2Rank — pocket predicted directly on the experimental structure

`<ds>_pdb_p2rank_sensitivity.tsv` (built in nb35) already picked, per source pocket, the single
best-DCA-matching P2Rank pocket among however many P2Rank returned for that structure — that's
`best_pocket`, with `dca`/`dcc`/`iou`/`recall` already computed against the true ligand.

In [3]:
rows = []
for ds in DATASETS:
    ids_p, arr_p = pickle.load(open(PROJ / f'RESULTS/{ds}/p2rank_pocket_encodings/pocket_reps.pkl', 'rb'))
    id2idx_p = {pid: i for i, pid in enumerate(ids_p)}
    sens = pd.read_csv(PROJ / f'RESULTS/benchmark/{ds}_pdb_p2rank_sensitivity.tsv', sep='\t')

    bp_ids, bp_reps = load_bound_pocket_reps(ds)
    bp_id2idx = {pid: i for i, pid in enumerate(bp_ids)}
    mol_mf, mol_reps, fold_masks = load_mol_reps(ds)
    mol_by_pocket = {r['pocket']: pos for pos, r in mol_mf.iterrows()}

    n_pairs = 0
    for _, r in sens.iterrows():
        pocket_id = r['pocket_id']
        bp_pred = r['best_pocket']
        if (pd.isna(bp_pred) or bp_pred not in id2idx_p
                or pocket_id not in mol_by_pocket or pocket_id not in bp_id2idx):
            continue
        pos = mol_by_pocket[pocket_id]
        lig_rep, lig_mask = mol_reps[pos], fold_masks[pos]
        pidx = id2idx_p[bp_pred]

        score_exp = float(cosine_scores(lig_rep, lig_mask, bp_reps[:, bp_id2idx[pocket_id]:bp_id2idx[pocket_id]+1, :])[0])
        score_pdb_p2rank = float(cosine_scores(lig_rep, lig_mask, arr_p[:, pidx:pidx+1, :])[0])
        pp_score = pocket_cosine(bp_reps[:, bp_id2idx[pocket_id], :], arr_p[:, pidx, :])

        rows.append({
            'dataset': ds, 'pocket_id': pocket_id, 'best_pocket_pred': bp_pred,
            'score_exp': score_exp, 'score_pdb_p2rank': score_pdb_p2rank,
            'pocket_pocket_score': pp_score,
            'dca': float(r['dca']), 'dcc': float(r['dcc']),
            'iou': float(r['iou']), 'recall': float(r['recall']),
            'n_obs': int(r['n_obs']), 'n_pred_res': int(r['n_pred_res']),
        })
        n_pairs += 1
    print(f'{ds}: {n_pairs} / {len(sens)} sensitivity rows scored')

pdb_p2rank_raw = pd.DataFrame(rows)
pdb_p2rank_raw['delta_score'] = pdb_p2rank_raw['score_pdb_p2rank'] - pdb_p2rank_raw['score_exp']
pdb_p2rank_raw['pdb_id'] = pdb_p2rank_raw['pocket_id'].map(mol_lookup['pdb_id'])
pdb_p2rank_raw['lig_resname'] = pdb_p2rank_raw['pocket_id'].map(mol_lookup['lig_resname'])
pdb_p2rank_raw['uniprot_acc'] = pdb_p2rank_raw['pocket_id'].map(uniprot_lookup)
pdb_p2rank_raw = pdb_p2rank_raw.dropna(subset=['pdb_id', 'lig_resname']).copy()

print(f'\nPDB P2Rank raw pooled: {len(pdb_p2rank_raw):,}')
pdb_p2rank = dedup_cross_dataset(pdb_p2rank_raw)
print(f'PDB P2Rank deduplicated: {len(pdb_p2rank):,} '
      f'({len(pdb_p2rank_raw) - len(pdb_p2rank):,} cross-dataset duplicates dropped)')
print(f'UniProt coverage: {pdb_p2rank.uniprot_acc.notna().sum():,} / {len(pdb_p2rank):,}')

out_path = OUT_DIR / 'pdb_p2rank_vs_exp_scores_pooled_dedup.tsv'
pdb_p2rank.to_csv(out_path, sep='\t', index=False)
print(f'Saved: {out_path}')
pdb_p2rank.head(3)

coach420: 323 / 330 sensitivity rows scored
holo4k: 6590 / 6599 sensitivity rows scored
pdbbind2020: 5091 / 5305 sensitivity rows scored
scpdb: 4956 / 4957 sensitivity rows scored

PDB P2Rank raw pooled: 16,960
PDB P2Rank deduplicated: 15,328 (1,632 cross-dataset duplicates dropped)
UniProt coverage: 12,749 / 15,328
Saved: /Users/jsutges/Documents/CHEMBL_DRUGCLIP/RESULTS/p2rank_patches_pooled/pdb_p2rank_vs_exp_scores_pooled_dedup.tsv


,dataset,pocket_id,best_pocket_pred,score_exp,score_pdb_p2rank,pocket_pocket_score,dca,dcc,iou,recall,n_obs,n_pred_res,delta_score,pdb_id,lig_resname,uniprot_acc
0,coach420,coach420_1a26_CNA_A_200,coach420__1a26A__p2rank_pocket1,-0.120696,-0.226779,0.505104,8.992561,11.131143,0.136364,0.315789,19,31,-0.106083,1a26,CNA,P26446
1,coach420,coach420_1a2k_GDP_C_220,coach420__1a2kC__p2rank_pocket1,0.846647,0.482083,0.555277,2.503280,2.379117,0.535714,0.576923,26,17,-0.364565,1a2k,GDP,P62825
2,coach420,coach420_1a4k_FRA_H_3083,coach420__1a4kH__p2rank_pocket1,0.464948,0.332682,0.556206,1.254890,1.635561,0.631579,0.666667,18,13,-0.132266,1a4k,FRA,NaN


## 2. AFDB P2Rank — pocket predicted on the AlphaFold model

`nb38_all_scores.tsv` supplies DCA / DCC / `p2r_rank` / `uniprot_acc` (pure geometry + P2Rank
metadata — independent of any embedding, so trustworthy regardless of how the scores below are
computed) for `strategy == 'best_dca'`: nb38's equivalent of PDB's `best_pocket` — the single
best-DCA-matching AFDB P2Rank pocket per structure.

**Scores are recomputed here from the raw embeddings, not read from nb38's cached `score_exp` /
`score_afdb_p2r` columns.** Spot-checking those columns against a fresh from-scratch cosine-score
computation (same pocket reps, same ligand reps, same formula, verified in two separate conda
environments) turned up large, systematic disagreement for a apparently large fraction of pockets —
e.g. `holo4k_1nn3_ADP_A_302` reads `score_exp = 0.835` in 37b's freshly-recomputed pooled table but
`score_exp = -0.491` in nb38's cached file, for the *identical* pocket/ligand pair and identical
scoring code. Cause not tracked down (files are unchanged since May; not an environment or dedup
artefact — ruled out both). Recomputing from the raw `.pkl`/`.h5` sources here sidesteps the
question entirely and keeps this condition self-consistent with PDB P2Rank and Patches above,
both of which are also computed fresh. **This same staleness risk applies to 37b's own saved
scores** (they were not recomputed as part of this notebook) — worth an independent audit.

In [4]:
afdb_p2r_all = pd.read_csv(PROJ / 'RESULTS/afdb_p2rank_pockets/nb38_all_scores.tsv', sep='\t')
afdb_p2r_meta = afdb_p2r_all[afdb_p2r_all['strategy'] == 'best_dca'][
    ['dataset', 'pocket_id', 'uniprot_acc', 'dca', 'dcc', 'p2r_rank', 'p2r_score',
     'p2r_probability', 'n_p2r_residues']
].copy()
print(f'AFDB P2Rank (best_dca strategy) metadata: {len(afdb_p2r_meta):,} rows')

p2r_ids, p2r_reps = pickle.load(open(PROJ / 'RESULTS/afdb_p2rank_pockets/pocket_reps.pkl', 'rb'))
p2r_id2idx = {pid: i for i, pid in enumerate(p2r_ids)}

rows = []
for ds in DATASETS:
    bp_ids, bp_reps = load_bound_pocket_reps(ds)
    bp_id2idx = {pid: i for i, pid in enumerate(bp_ids)}
    mol_mf, mol_reps, fold_masks = load_mol_reps(ds)
    mol_by_pocket = {r['pocket']: pos for pos, r in mol_mf.iterrows()}

    ds_meta = afdb_p2r_meta[afdb_p2r_meta['dataset'] == ds]
    n_pairs = 0
    for _, m in ds_meta.iterrows():
        pocket_id = m['pocket_id']
        if pocket_id not in mol_by_pocket or pocket_id not in bp_id2idx or pd.isna(m['p2r_rank']):
            continue
        p2r_key = f"afdb_p2rank__{m['uniprot_acc']}__pocket{int(m['p2r_rank'])}"
        p2r_idx = p2r_id2idx.get(p2r_key)
        if p2r_idx is None:
            continue

        pos = mol_by_pocket[pocket_id]
        lig_rep, lig_mask = mol_reps[pos], fold_masks[pos]
        bp_idx = bp_id2idx[pocket_id]

        score_exp = float(cosine_scores(lig_rep, lig_mask, bp_reps[:, bp_idx:bp_idx+1, :])[0])
        score_afdb_p2rank = float(cosine_scores(lig_rep, lig_mask, p2r_reps[:, p2r_idx:p2r_idx+1, :])[0])
        pp_score = pocket_cosine(bp_reps[:, bp_idx, :], p2r_reps[:, p2r_idx, :])

        rows.append({
            'dataset': ds, 'pocket_id': pocket_id, 'uniprot_acc': m['uniprot_acc'],
            'score_exp': score_exp, 'score_afdb_p2rank': score_afdb_p2rank,
            'pocket_pocket_score': pp_score,
            'dca': float(m['dca']), 'dcc': float(m['dcc']),
            'p2r_rank': int(m['p2r_rank']), 'p2r_score': float(m['p2r_score']),
            'p2r_probability': float(m['p2r_probability']),
            'n_p2r_residues': None if pd.isna(m['n_p2r_residues']) else float(m['n_p2r_residues']),
        })
        n_pairs += 1
    print(f'{ds}: {n_pairs} / {len(ds_meta)} AFDB P2Rank rows scored (fresh)')

afdb_p2r_raw = pd.DataFrame(rows)
afdb_p2r_raw['delta_score'] = afdb_p2r_raw['score_afdb_p2rank'] - afdb_p2r_raw['score_exp']
afdb_p2r_raw['pdb_id'] = afdb_p2r_raw['pocket_id'].map(mol_lookup['pdb_id'])
afdb_p2r_raw['lig_resname'] = afdb_p2r_raw['pocket_id'].map(mol_lookup['lig_resname'])
afdb_p2r_raw = afdb_p2r_raw.dropna(subset=['pdb_id', 'lig_resname']).copy()

print(f'\nAFDB P2Rank raw pooled: {len(afdb_p2r_raw):,}')
print(f'median score_exp (sanity check — should be ~0.6-0.75, matching 37b/45/46): '
      f'{afdb_p2r_raw.score_exp.median():.4f}')
afdb_p2rank = dedup_cross_dataset(afdb_p2r_raw)
print(f'AFDB P2Rank deduplicated: {len(afdb_p2rank):,} '
      f'({len(afdb_p2r_raw) - len(afdb_p2rank):,} cross-dataset duplicates dropped)')
print(f'UniProt coverage: {afdb_p2rank.uniprot_acc.notna().sum():,} / {len(afdb_p2rank):,}')

out_path = OUT_DIR / 'afdb_p2rank_vs_exp_scores_pooled_dedup.tsv'
afdb_p2rank.to_csv(out_path, sep='\t', index=False)
print(f'Saved: {out_path}')
afdb_p2rank.head(3)

AFDB P2Rank (best_dca strategy) metadata: 10,780 rows
coach420: 235 / 235 AFDB P2Rank rows scored (fresh)
holo4k: 4106 / 4106 AFDB P2Rank rows scored (fresh)
pdbbind2020: 3617 / 3617 AFDB P2Rank rows scored (fresh)
scpdb: 2822 / 2822 AFDB P2Rank rows scored (fresh)

AFDB P2Rank raw pooled: 10,780
median score_exp (sanity check — should be ~0.6-0.75, matching 37b/45/46): 0.7108
AFDB P2Rank deduplicated: 9,831 (949 cross-dataset duplicates dropped)
UniProt coverage: 9,831 / 9,831
Saved: /Users/jsutges/Documents/CHEMBL_DRUGCLIP/RESULTS/p2rank_patches_pooled/afdb_p2rank_vs_exp_scores_pooled_dedup.tsv


,dataset,pocket_id,uniprot_acc,score_exp,score_afdb_p2rank,pocket_pocket_score,dca,dcc,p2r_rank,p2r_score,p2r_probability,n_p2r_residues,delta_score,pdb_id,lig_resname
0,coach420,coach420_1a26_CNA_A_200,P26446,-0.120696,-0.192861,0.392477,8.797403,11.061549,1,17.91,0.810,32.0,-0.072166,1a26,CNA
1,coach420,coach420_1a2k_GDP_C_220,P62825,0.846647,0.699318,0.697435,0.637383,4.771669,1,8.17,0.435,22.0,-0.147329,1a2k,GDP
2,coach420,coach420_1a7x_FKA_B_201,P62942,0.095600,0.227206,-0.024431,8.618873,8.053220,1,5.87,0.287,12.0,0.131606,1a7x,FKA


## 2b. AFDB P2Rank — residue-level IoU / recall

nb38 computed this internally (`ov_all`) but never saved it — only DCA/DCC made it into
`nb38_all_scores.tsv`. All the pieces to redo it are on disk and already mutually consistent, so
this recomputes it directly rather than re-deriving anything nb38 didn't already solve:

- **Predicted residues**: `RESULTS/afdb_p2rank/AF-<uniprot>-F1-model_v6.cif_predictions.csv` — P2Rank's
  raw per-pocket `residue_ids` column, in AFDB numbering (`auth_seq_id == label_seq_id` for an AFDB
  model — no icode/insertion complications).
- **PDB ↔ AFDB residue mapping**: `aln_map_cache.pkl`, keyed by `(entry, uniprot_acc, lig_chain)` —
  the exact sequence-alignment map nb38 built (and cached) to transform ligand coordinates into the
  AFDB frame for DCA/DCC. Maps the predicted residues into PDB numbering.
- **True/observed residues**: `pocket_manifest.tsv`'s `pocket_residues` column — the same 6 Å
  ligand-contact residue set used to build every pocket embedding in this whole project (verified:
  its residue count matches nb35's `n_obs` for spot-checked pockets exactly).

IoU = |obs ∩ pred| / |obs ∪ pred|, recall = |obs ∩ pred| / |obs| — same formulas nb35 uses for PDB
P2Rank, so the two conditions' IoU/recall are directly comparable.

In [5]:
DS_KEY_FOR_POCKETS = {'coach420': 'coach420', 'holo4k': 'holo4k',
                      'pdbbind2020': 'pdbbind2020', 'scpdb': 'sc-pdb'}
_RES_ID_RE = re.compile(r'[A-Za-z0-9]+_(-?\d+)')

aln_map_cache = pickle.load(open(PROJ / 'RESULTS/afdb_p2rank_pockets/aln_map_cache.pkl', 'rb'))

pocket_meta = {}
for ds in DATASETS:
    raw = pickle.load(open(NL / 'pockets' / f'{ds}_pockets.pkl', 'rb'))
    for pid, v in raw.items():
        pocket_meta[pid] = {'entry': v['_entry'], 'lig_chain': v.get('_lig_chain', 'A')}

obs_residues = {}
for ds in DATASETS:
    mf = pd.read_csv(NL / 'pockets' / DS_KEY_FOR_POCKETS[ds] / 'pocket_manifest.tsv', sep='\t')
    for _, row in mf.iterrows():
        try:
            obs_residues[row['pocket']] = frozenset(int(r[1]) for r in json.loads(row['pocket_residues']))
        except Exception:
            pass

_p2r_res_cache: dict[str, dict] = {}
def load_p2r_residues(acc: str) -> dict:
    if acc in _p2r_res_cache:
        return _p2r_res_cache[acc]
    csv_path = PROJ / f'RESULTS/afdb_p2rank/AF-{acc}-F1-model_v6.cif_predictions.csv'
    out = {}
    if csv_path.exists():
        df_csv = pd.read_csv(csv_path)
        df_csv.columns = [c.strip() for c in df_csv.columns]
        for _, row in df_csv.iterrows():
            name = str(row['name']).strip()
            out[name] = frozenset(int(m.group(1)) for m in _RES_ID_RE.finditer(str(row['residue_ids'])))
    _p2r_res_cache[acc] = out
    return out

iou_rows, missing = [], {'aln': 0, 'pred': 0, 'obs': 0}
for _, r in afdb_p2rank.iterrows():
    pocket_id, uniprot, p2r_rank = r['pocket_id'], r['uniprot_acc'], r['p2r_rank']
    pm = pocket_meta.get(pocket_id)
    if pm is None or pd.isna(p2r_rank):
        missing['aln'] += 1; continue
    aln_map = aln_map_cache.get((pm['entry'], uniprot, pm['lig_chain']))
    if not aln_map:
        missing['aln'] += 1; continue
    pred_afdb_ids = load_p2r_residues(uniprot).get(f'pocket{int(p2r_rank)}', frozenset())
    if not pred_afdb_ids:
        missing['pred'] += 1; continue
    pred_pdb_rset = frozenset(aln_map[x] for x in pred_afdb_ids if x in aln_map)
    obs_rset = obs_residues.get(pocket_id)
    if not obs_rset or not pred_pdb_rset:
        missing['obs'] += 1; continue

    inter = len(obs_rset & pred_pdb_rset)
    iou_rows.append({
        'pocket_id': pocket_id,
        'iou': inter / len(obs_rset | pred_pdb_rset),
        'recall': inter / len(obs_rset),
        'n_obs_residues': len(obs_rset), 'n_pred_pdb_residues': len(pred_pdb_rset),
    })

iou_tbl = pd.DataFrame(iou_rows).set_index('pocket_id')
print(f'IoU/recall computed: {len(iou_tbl):,} / {len(afdb_p2rank):,}  '
      f'(missing: no alignment={missing["aln"]}, no predicted residues={missing["pred"]}, '
      f'no observed residues={missing["obs"]})')

afdb_p2rank = afdb_p2rank.join(iou_tbl, on='pocket_id')
print(f'\nIoU:    {afdb_p2rank["iou"].describe(percentiles=[.25,.5,.75]).round(3).to_dict()}')
print(f'Recall: {afdb_p2rank["recall"].describe(percentiles=[.25,.5,.75]).round(3).to_dict()}')

out_path = OUT_DIR / 'afdb_p2rank_vs_exp_scores_pooled_dedup.tsv'
afdb_p2rank.to_csv(out_path, sep='\t', index=False)
print(f'Re-saved with iou/recall: {out_path}')
afdb_p2rank.head(3)

/var/folders/7g/mgz14fdd72sb7p9dl4w7m24h0000gn/T/ipykernel_64009/2446500606.py:9: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  raw = pickle.load(open(NL / 'pockets' / f'{ds}_pockets.pkl', 'rb'))


IoU/recall computed: 9,815 / 9,831  (missing: no alignment=0, no predicted residues=0, no observed residues=16)

IoU:    {'count': 9815.0, 'mean': 0.457, 'std': 0.184, 'min': 0.0, '25%': 0.346, '50%': 0.477, '75%': 0.591, 'max': 0.952}
Recall: {'count': 9815.0, 'mean': 0.571, 'std': 0.222, 'min': 0.0, '25%': 0.44, '50%': 0.615, '75%': 0.731, 'max': 1.0}
Re-saved with iou/recall: /Users/jsutges/Documents/CHEMBL_DRUGCLIP/RESULTS/p2rank_patches_pooled/afdb_p2rank_vs_exp_scores_pooled_dedup.tsv


,dataset,pocket_id,uniprot_acc,score_exp,score_afdb_p2rank,pocket_pocket_score,dca,dcc,p2r_rank,p2r_score,p2r_probability,n_p2r_residues,delta_score,pdb_id,lig_resname,iou,recall,n_obs_residues,n_pred_pdb_residues
0,coach420,coach420_1a26_CNA_A_200,P26446,-0.120696,-0.192861,0.392477,8.797403,11.061549,1,17.91,0.810,32.0,-0.072166,1a26,CNA,0.133333,0.315789,19.0,32.0
1,coach420,coach420_1a2k_GDP_C_220,P62825,0.846647,0.699318,0.697435,0.637383,4.771669,1,8.17,0.435,22.0,-0.147329,1a2k,GDP,0.371429,0.500000,26.0,22.0
2,coach420,coach420_1a7x_FKA_B_201,P62942,0.095600,0.227206,-0.024431,8.618873,8.053220,1,5.87,0.287,12.0,0.131606,1a7x,FKA,0.238095,0.357143,14.0,12.0


## 3. Random patches — no detection involved

Every source pocket's structure has multiple random surface patches (nb30/31); the ligand's score
is **mean-pooled** across all of them, same convention as everywhere else multi-counterpart
conditions come up in this project. No DCA/DCC/IoU here — a random patch isn't "detecting" anything,
so there's no ground truth to score it against; only the score distribution itself is meaningful.

In [6]:
rows = []
for ds in DATASETS:
    ids_patch, arr_patch = pickle.load(open(PROJ / f'RESULTS/{ds}/pocket_reps.pkl', 'rb'))
    entry2patch: defaultdict = defaultdict(list)
    for j, pid in enumerate(ids_patch):
        entry2patch[pid.split('__')[1]].append(j)

    bp_ids, bp_reps = load_bound_pocket_reps(ds)
    bp_id2idx = {pid: i for i, pid in enumerate(bp_ids)}
    mol_mf, mol_reps, fold_masks = load_mol_reps(ds)

    n_pairs = 0
    for pos, mol_row in mol_mf.iterrows():
        pocket_id = mol_row['pocket']
        idxs = entry2patch.get(mol_row['entry'], [])
        if not idxs or pocket_id not in bp_id2idx:
            continue
        lig_rep, lig_mask = mol_reps[pos], fold_masks[pos]

        score_exp = float(cosine_scores(lig_rep, lig_mask, bp_reps[:, bp_id2idx[pocket_id]:bp_id2idx[pocket_id]+1, :])[0])
        patch_scores = cosine_scores(lig_rep, lig_mask, arr_patch[:, idxs, :])  # mean-pooled below

        rows.append({
            'dataset': ds, 'pocket_id': pocket_id,
            'score_exp': score_exp, 'score_patches': float(np.mean(patch_scores)),
            'n_patches': len(idxs),
        })
        n_pairs += 1
    print(f'{ds}: {n_pairs} / {len(mol_mf)} pockets scored (mean-pooled over patches)')

patches_raw = pd.DataFrame(rows)
patches_raw['delta_score'] = patches_raw['score_patches'] - patches_raw['score_exp']
patches_raw['pdb_id'] = patches_raw['pocket_id'].map(mol_lookup['pdb_id'])
patches_raw['lig_resname'] = patches_raw['pocket_id'].map(mol_lookup['lig_resname'])
patches_raw['uniprot_acc'] = patches_raw['pocket_id'].map(uniprot_lookup)
patches_raw = patches_raw.dropna(subset=['pdb_id', 'lig_resname']).copy()

print(f'\nPatches raw pooled: {len(patches_raw):,}')
print(f'Counterparts per pocket: median={patches_raw.n_patches.median():.0f}  max={patches_raw.n_patches.max()}')
patches = dedup_cross_dataset(patches_raw)
print(f'Patches deduplicated: {len(patches):,} '
      f'({len(patches_raw) - len(patches):,} cross-dataset duplicates dropped)')
print(f'UniProt coverage: {patches.uniprot_acc.notna().sum():,} / {len(patches):,}')

out_path = OUT_DIR / 'patches_vs_exp_scores_pooled_dedup.tsv'
patches.to_csv(out_path, sep='\t', index=False)
print(f'Saved: {out_path}')
patches.head(3)

coach420: 335 / 335 pockets scored (mean-pooled over patches)
holo4k: 6690 / 6690 pockets scored (mean-pooled over patches)
pdbbind2020: 5313 / 5313 pockets scored (mean-pooled over patches)
scpdb: 4967 / 4967 pockets scored (mean-pooled over patches)

Patches raw pooled: 17,305
Counterparts per pocket: median=16  max=124
Patches deduplicated: 15,640 (1,665 cross-dataset duplicates dropped)
UniProt coverage: 13,014 / 15,640
Saved: /Users/jsutges/Documents/CHEMBL_DRUGCLIP/RESULTS/p2rank_patches_pooled/patches_vs_exp_scores_pooled_dedup.tsv


,dataset,pocket_id,score_exp,score_patches,n_patches,delta_score,pdb_id,lig_resname,uniprot_acc
0,coach420,coach420_1a26_CNA_A_200,-0.120696,0.00531,20,0.126006,1a26,CNA,P26446
1,coach420,coach420_1a2k_GDP_C_220,0.846647,-0.10102,10,-0.947668,1a2k,GDP,P62825
2,coach420,coach420_1br6_PT1_A_301,0.721427,0.14856,13,-0.572867,1br6,PT1,P02879


In [7]:
patches

,dataset,pocket_id,score_exp,score_patches,n_patches,delta_score,pdb_id,lig_resname,uniprot_acc
0,coach420,coach420_1a26_CNA_A_200,-0.120696,0.005310,20,0.126006,1a26,CNA,P26446
1,coach420,coach420_1a2k_GDP_C_220,0.846647,-0.101020,10,-0.947668,1a2k,GDP,P62825
2,coach420,coach420_1br6_PT1_A_301,0.721427,0.148560,13,-0.572867,1br6,PT1,P02879
3,coach420,coach420_1n07_ADP_A_164,0.754236,-0.066611,10,-0.820847,1n07,ADP,O74866
4,coach420,coach420_1n07_FMN_A_165,0.568140,0.042646,10,-0.525493,1n07,FMN,O74866
...,...,...,...,...,...,...,...,...,...
17300,scpdb,sc-pdb_2a4f_AAU_A_1,0.772305,0.070006,9,-0.702298,2a4f,AAU,NaN
17301,scpdb,sc-pdb_2a4x_BLM_A_1,0.337525,0.162474,13,-0.175051,2a4x,BLM,NaN
17302,scpdb,sc-pdb_2a59_LMZ_A_1,0.698934,-0.065877,13,-0.764810,2a59,LMZ,Q9UUB1
17303,scpdb,sc-pdb_2a5h_SAM_A_1,0.412387,0.008906,22,-0.403482,2a5h,SAM,NaN


## 4. Summary across the three conditions

In [6]:
summary = pd.DataFrame([
    {
        'condition': 'PDB P2Rank', 'n': len(pdb_p2rank),
        'n_unique_uniprot': pdb_p2rank.uniprot_acc.nunique(),
        'median_score_exp': pdb_p2rank.score_exp.median(),
        'median_score_target': pdb_p2rank.score_pdb_p2rank.median(),
        'median_delta': pdb_p2rank.delta_score.median(),
        'pct_drop': (pdb_p2rank.delta_score < 0).mean() * 100,
        'median_dca': pdb_p2rank.dca.median(),
    },
    {
        'condition': 'AFDB P2Rank', 'n': len(afdb_p2rank),
        'n_unique_uniprot': afdb_p2rank.uniprot_acc.nunique(),
        'median_score_exp': afdb_p2rank.score_exp.median(),
        'median_score_target': afdb_p2rank.score_afdb_p2rank.median(),
        'median_delta': afdb_p2rank.delta_score.median(),
        'pct_drop': (afdb_p2rank.delta_score < 0).mean() * 100,
        'median_dca': afdb_p2rank.dca.median(),
    },
    {
        'condition': 'Patches', 'n': len(patches),
        'n_unique_uniprot': patches.uniprot_acc.nunique(),
        'median_score_exp': patches.score_exp.median(),
        'median_score_target': patches.score_patches.median(),
        'median_delta': patches.delta_score.median(),
        'pct_drop': (patches.delta_score < 0).mean() * 100,
        'median_dca': np.nan,
    },
]).set_index('condition').round(4)

print(summary.to_string())
summary

                 n  n_unique_uniprot  median_score_exp  median_score_target  median_delta  pct_drop  median_dca
condition                                                                                                      
PDB P2Rank   15328              4827            0.7134               0.1860       -0.4264   95.4854      1.7144
AFDB P2Rank   9831              3817            0.7064               0.1020       -0.5112   96.9383      1.6488
Patches      15640              4869            0.7129               0.0014       -0.6842   98.6253         NaN


,n,n_unique_uniprot,median_score_exp,median_score_target,median_delta,pct_drop,median_dca
condition,,,,,,,
PDB P2Rank,15328,4827,0.7134,0.1860,-0.4264,95.4854,1.7144
AFDB P2Rank,9831,3817,0.7064,0.1020,-0.5112,96.9383,1.6488
Patches,15640,4869,0.7129,0.0014,-0.6842,98.6253,NaN
